# Reflection and Blogpost Writing

## Setup

In [1]:
#from utils import get_openai_api_key
#OPENAI_API_KEY = get_openai_api_key()
#print(OPENAI_API_KEY)
#llm_config = {"model": "gpt-3.5-turbo","api_key":OPENAI_API_KEY}

#llm_config = {"model": "llama3","api_type":"ollama"}

llm_config = {"model": "gemini-3.5-flash","api_key":"","api_type":"google"}


## The task!

In [2]:
task = '''
        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       '''


## Create a writer agent

In [3]:
import autogen

writer = autogen.AssistantAgent(
    name="Writer",
    system_message="You are a writer. You write engaging and concise " 
        "blogpost (with title) on given topics. You must polish your "
        "writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments.",
    llm_config=llm_config,
)

C:\Users\User\anaconda3\Lib\site-packages\autogen\oai\gemini.py:65: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Patching name='__init__', member=<function BedrockClient.__init__ at 0x0000026606DC1B20>, patched=<function function.__call__ at 0x0000026606DC1A80>
Patching name='_retries', member=5, patched=5
Patching name='cost', member=<function BedrockClient.cost at 0x0000026606DC1E40>, patched=<function function.__call__ at 0x0000026606DC20C0>
Patching name='create', member=<function BedrockClient.create at 0x0000026606DC1DA0>, patched=<function function.__call__ at 0x0000026606DC2160>
Patching name='get_usage', member=<function BedrockClient.get_usage at 0x0000026606DC1EE0>, patched=<function function.__call__ at 0x0000026606DC2200>
Patching name='message_retrieval', member=<function BedrockClient.message_retrieval at 0x0000026606DC1BC0>, patched=<function function.__call__ at 0x0000026606DC22A0>
Patching name='parse_custom_params', member=<function BedrockClient.parse_custom_params at 0x0000026606DC1C60>, patched=<function function.__call__ at 0x0000026606DC2340>
Patching name='parse_params', 

In [4]:
reply = writer.generate_reply(messages=[{"content": task ,"role": "user"}])

In [5]:
print(reply)

{'content': '**Hope AI: Coding a Brighter Tomorrow**\n\nWhat if technology’s greatest feat isn’t intelligence, but empathy? Enter Hope AI—a movement transforming algorithms into forces for global good. \n\nFar from the cold, robotic future we often fear, Hope AI is designed to solve humanity’s deepest crises. It’s predicting climate disasters, accelerating medical breakthroughs, and providing accessible mental health support. By marrying vast computing power with human-centric goals, Hope AI turns despair into actionable solutions. \n\nThe future isn’t a sci-fi dystopia. With Hope AI, we are rewriting the code of tomorrow, one breakthrough at a time. Optimism is officially online.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}


## Adding reflection 

Create a critic agent to reflect on the work of the writer agent.

In [6]:
critic = autogen.AssistantAgent(
    name="Critic",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
    system_message="You are a critic. You review the work of "
                "the writer and provide constructive "
                "feedback to help improve the quality of the content.",
)

In [7]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

**Coding Optimism: The Rise of Hope AI**

Fear of the future is outdated. Today, a new wave of technology—Hope AI—is proving that artificial intelligence can be humanity's ultimate lifesaver. 

Instead of replacing us, Hope AI tackles our biggest crises. It’s mapping climate change, diagnosing rare diseases in seconds, and optimizing global food supplies. By aligning machine learning with human empathy, we are turning data into destiny. 

This isn't just about smarter software; it’s about a more compassionate world. The future isn’t a threat—it’s a promise, written in code. Welcome to the era of Hope AI.

--------------------------------------------------------------------------------
Critic (to Writer):

Here is my critique of your blog 

## Nested chat

In [8]:
SEO_reviewer = autogen.AssistantAgent(
    name="SEO_Reviewer",
    llm_config=llm_config,
    system_message="You are an SEO reviewer, known for "
        "your ability to optimize content for search engines, "
        "ensuring that it ranks well and attracts organic traffic. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)


In [9]:
legal_reviewer = autogen.AssistantAgent(
    name="Legal_Reviewer",
    llm_config=llm_config,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [10]:
ethics_reviewer = autogen.AssistantAgent(
    name="Ethics_Reviewer",
    llm_config=llm_config,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [11]:
meta_reviewer = autogen.AssistantAgent(
    name="Meta_Reviewer",
    llm_config=llm_config,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

## Orchestrate the nested chats to solve the task

In [12]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [13]:
critic.register_nested_chats(
    review_chats,
    trigger=writer,
)

**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [14]:
res = critic.initiate_chat(
    recipient=writer,
    message=task,
    max_turns=2,
    summary_method="last_msg"
)

Critic (to Writer):


        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

--------------------------------------------------------------------------------
Writer (to Critic):

**Coding Optimism: The Rise of Hope AI**

Fear of the future is outdated. Today, a new wave of technology—Hope AI—is proving that artificial intelligence can be humanity's ultimate lifesaver. 

Instead of replacing us, Hope AI tackles our biggest crises. It’s mapping climate change, diagnosing rare diseases in seconds, and optimizing global food supplies. By aligning machine learning with human empathy, we are turning data into destiny. 

This isn't just about smarter software; it’s about a more compassionate world. The future isn’t a threat—it’s a promise, written in code. Welcome to the era of Hope AI.

--------------------------------------------------------------------------------

*****************************************************

C:\Users\User\anaconda3\Lib\site-packages\autogen\oai\gemini.py:803: UserWarning: Cost calculation is not implemented for model gemini-3.5-flash. Cost will be calculated zero.
  warnings.warn(


Writer (to Critic):

**Coding Optimism: How AI for Social Good is Shaping the Future**

Instead of a magic savior, collaborative AI is a tool for human progress. By pairing machine learning with human expertise, researchers aim to better analyze climate patterns, support clinicians identifying rare diseases, and assist farmers optimizing crop yields. 

This technology doesn't replace human empathy or judgment; it equips experts with data-driven insights. Developed and used responsibly, AI for social good is helping us build a safer, more sustainable tomorrow.

--------------------------------------------------------------------------------


## Get the summary

In [15]:
print(res.summary)

**Coding Optimism: How AI for Social Good is Shaping the Future**

Instead of a magic savior, collaborative AI is a tool for human progress. By pairing machine learning with human expertise, researchers aim to better analyze climate patterns, support clinicians identifying rare diseases, and assist farmers optimizing crop yields. 

This technology doesn't replace human empathy or judgment; it equips experts with data-driven insights. Developed and used responsibly, AI for social good is helping us build a safer, more sustainable tomorrow.
